### Обязательные библиотеки:
- [Gensim](https://radimrehurek.com/gensim/) — инструмент для решения различных задач NLP (тематическое моделирование, представление текстов, ...);
- [Numpy](http://www.numpy.org) — библиотека для векторных вычислений;
Для выполнения дополнительных заданий (по желанию)
- [Pandas](https://pandas.pydata.org) - библиотека для анализа табличных данных;
- [scikit-learn](http://scikit-learn.org/stable/index.html) — библилиотека с алгоритмами классического машинного обучения;
- [matplotlib](https://matplotlib.org) - библиотека для построения графиков;

-------

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

# Константы
DATA_DIR = 'data'
RANDOM_STATE = 42

### Модель, генерирующая вектора слов

In [ ]:
from utils import download_model

download_model(data_dir=DATA_DIR)

In [ ]:
import gensim

wv_embeddings = gensim.models.fasttext.FastTextKeyedVectors.load(
    DATA_DIR + "/model/model.model"
)

---

In [ ]:
# =============================================================================
# ЗАДАНИЕ 1: Исследование эмбеддингов
# =============================================================================

print("=== ЗАДАНИЕ 1: Исследование эмбеддингов ===")

# Проверяем размерность эмбеддинга
print("Размерность эмбеддинга:", len(wv_embeddings['студент']))
print("\nПример эмбеддинга слова 'студент' (первые 10 значений):")
print(wv_embeddings['студент'][:10])

# Проверяем различные типы слов
test_words = [
    'студент',           # обычное слово
    'стидент',          # слово с опечаткой
    'авадакедавра',          # несуществующее слово
    'мгу',             # аббревиатура
    'фыокаидывоп',        # случайный набор букв
    'эмбеддинг',         # современное слово
    '42',             # цифры
    'print'              # английское слово
]

print("\nПроверка работы с разными типами слов:")
for word in test_words:
    try:
        emb = wv_embeddings[word]
        print(f"✓ '{word}' - эмбеддинг получен (длина: {len(emb)})")
    except KeyError:
        print(f"✗ '{word}' - слово не найдено")
    except Exception as e:
        print(f"✗ '{word}' - ошибка: {e}")

# Нормировка эмбеддинга
print("\nНормированный эмбеддинг 'студент' (первые 10 значений):")
emb = np.array(wv_embeddings['студент'])
emb /= np.linalg.norm(emb)
print(emb[:10])

# =============================================================================
# ЗАДАНИЕ 2: Метод most_similar для частых и редких слов
# =============================================================================

print("\n\n=== ЗАДАНИЕ 2: Метод most_similar ===")

# Частоупотребляемые слова
common_words = ['город', 'работа', 'время', 'человек', 'дом']
print("\nЧастоупотребляемые слова:")
for word in common_words:
    try:
        similar = wv_embeddings.most_similar(word, topn=5)
        print(f"\nСлово '{word}':")
        for sim_word, score in similar:
            print(f"  {sim_word}: {score:.3f}")
    except Exception as e:
        print(f"Ошибка для слова '{word}': {e}")

# Редкие и специфические слова
rare_words = ['паребрик', 'когерентный', 'абракадабра', 'векторный', 'трансформер']
print("\nРедкие и специфические слова:")
for word in rare_words:
    try:
        similar = wv_embeddings.most_similar(word, topn=5)
        print(f"\nСлово '{word}':")
        for sim_word, score in similar:
            print(f"  {sim_word}: {score:.3f}")
    except Exception as e:
        print(f"Ошибка для слова '{word}': {e}")

# =============================================================================
# ЗАДАНИЕ 3: Другие методы - предсказуемые и непредсказуемые примеры
# =============================================================================

print("\n\n=== ЗАДАНИЕ 3: Другие методы ===")

# Метод most_similar_to_given
print("\n1. Метод most_similar_to_given:")

# Предсказуемый пример
result1 = wv_embeddings.most_similar_to_given('кошка', ['собака', 'дерево', 'камень', 'автомобиль'])
print(f"Предсказуемый: 'кошка' → {result1}")

# Непредсказуемый пример
result2 = wv_embeddings.most_similar_to_given('программирование', ['искусство', 'математика', 'литература', 'спорт'])
print(f"Непредсказуемый: 'программирование' → {result2}")

# Еще один пример
result3 = wv_embeddings.most_similar_to_given('любовь', ['ненависть', 'радость', 'печаль', 'гнев'])
print(f"Эмоции: 'любовь' → {result3}")

# Метод doesnt_match
print("\n2. Метод doesnt_match:")

# Предсказуемый пример
result4 = wv_embeddings.doesnt_match(['яблоко', 'груша', 'слива', 'автомобиль', 'вишня'])
print(f"Предсказуемый: лишнее слово - '{result4}'")

# Непредсказуемый пример
result5 = wv_embeddings.doesnt_match(['любовь', 'ненависть', 'радость', 'печаль', 'гнев'])
print(f"Непредсказуемый: лишнее слово - '{result5}'")

# Еще один пример
result6 = wv_embeddings.doesnt_match(['стол', 'стул', 'кровать', 'диван', 'компьютер'])
print(f"Мебель: лишнее слово - '{result6}'")

# Метод most_similar с positive/negative
print("\n3. Метод most_similar с positive/negative:")

# Классический пример: король - мужчина + женщина = королева
print("Классический пример (король - мужчина + женщина):")
result7 = wv_embeddings.most_similar(
    positive=['король', 'женщина'], 
    negative=['мужчина'],
    topn=3
)
for word, score in result7:
    print(f"  {word}: {score:.3f}")

# Непредсказуемый пример
print("\nНепредсказуемый пример (программист - код + музыка):")
result8 = wv_embeddings.most_similar(
    positive=['программист', 'музыка'], 
    negative=['код'],
    topn=3
)
for word, score in result8:
    print(f"  {word}: {score:.3f}")

# Еще один интересный пример
print("\nПример с профессиями (врач - больной + ученик):")
result9 = wv_embeddings.most_similar(
    positive=['врач', 'ученик'], 
    negative=['больной'],
    topn=3
)
for word, score in result9:
    print(f"  {word}: {score:.3f}")

# Дополнительные эксперименты с most_similar
print("\n4. Дополнительные эксперименты:")

# Многозначное слово
print("Многозначное слово 'модель':")
result10 = wv_embeddings.most_similar('модель', topn=8)
for word, score in result10:
    print(f"  {word}: {score:.3f}")

# Слово с опечаткой
print("\nСлово с опечаткой 'превет':")
try:
    result11 = wv_embeddings.most_similar('превет', topn=5)
    for word, score in result11:
        print(f"  {word}: {score:.3f}")
except Exception as e:
    print(f"  Ошибка: {e}")

print("\n=== ЭКСПЕРИМЕНТЫ ЗАВЕРШЕНЫ ===")